# Step-by-step process for applying PEFT to a pretrained model
This reading will guide you through the following steps:

Step 1: Prepare your dataset

Step 2: Fine-tune the pretrained model

Step 3: Evaluate the model

Step 4: Optimize the fine-tuning process (optional)

# Step 1: Prepare your dataset
The first step in fine-tuning a model is ensuring your dataset is appropriately prepared. In this walkthrough, we’ll follow the same steps from the activity to clean, tokenize, and split the dataset into training, validation, and test sets. This helps avoid overfitting and ensures proper model evaluation.

# Instructions
Load the dataset and inspect its structure.

Split the dataset into training, validation, and test sets to avoid overfitting and ensure good generalization.

Preprocess the data by cleaning, tokenizing, and padding the text.

Code example

### Recommended Dataset: AG News

For a text classification task involving fine-tuning a BERT model, a suitable dataset is the **AG News** dataset. It consists of news articles categorized into four classes: World, Sports, Business, and Sci/Tech. This dataset is commonly used for benchmarking text classification models.

In [ ]:
# Install the `datasets` library to easily load the AG News dataset
!pip install datasets

In [ ]:
# Ensure necessary libraries are installed or upgraded with explicit version pinning and reinstallation
# Uninstall potentially conflicting packages first for a clean slate
!pip uninstall -y pyarrow datasets huggingface_hub pandas

# Install specific, known compatible versions that support Python 3.12
!pip install pyarrow==16.0.0 # Updated to a version with Python 3.12 wheels
!pip install datasets # Install latest compatible version
!pip install huggingface_hub # Install latest compatible version
!pip install pandas # Install latest compatible version

from datasets import load_dataset
import pandas as pd

# Load the AG News dataset from the Hugging Face Hub
# The 'ag_news' dataset provides 'text' and 'label' columns, and labels are typically 0-indexed.
dataset = load_dataset('ag_news', split='train')

# Convert to pandas DataFrame for easier manipulation
df_agnews = dataset.to_pandas()

# The 'text' column already contains the combined title and description.
# The 'label' column is already 0-indexed (0, 1, 2, 3), so no adjustment is needed.

# Inspect the first few rows
display(df_agnews.head())

# Save the processed dataframe to a CSV file (this will be used by the splitting cell)
df_agnews[['text', 'label']].to_csv('ag_news_dataset.csv', index=False)
print("AG News dataset processed and saved as 'ag_news_dataset.csv'")

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset (using the AG News dataset saved previously)
data = pd.read_csv('ag_news_dataset.csv')

# Split dataset into training (70%), validation (15%), and test (15%)
train_data, temp_data = train_test_split(data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Reset indices to ensure contiguous integer indexing for the Trainer
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

Training set size: 84000
Validation set size: 18000
Test set size: 18000


In [7]:
import pandas as pd

data = pd.read_csv('ag_news_dataset.csv')

print(data.columns.tolist())
print(data['label'].value_counts())  # or 'labels', 'class', etc.

['text', 'label']
label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64


# Explanation
Here, we use train_test_split to divide the dataset into three parts: 70 percent for training, 15 percent for validation, and 15 percent for testing. This is accomplished by re-splitting the temp_data output from the first call of train_test_split. The training set is used to fine-tune the model, the validation set helps to monitor training progress, and the test set evaluates the final model performance.

#Step 2: Fine-tune the pretrained model
Once the data is prepared, the next step is to fine-tune the pretrained model (e.g., BERT) using the training data. Fine-tuning involves adapting the model’s existing knowledge to a new, task-specific dataset.

Instructions
Load a pretrained model (e.g., BERT).

Set up the training arguments, such as batch size, learning rate, and number of epochs.

Begin fine-tuning the model on the training set.

Code example

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import Dataset # Import Dataset from datasets library

# Load pretrained BERT model with the correct number of labels (4 for AG News)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Convert pandas DataFrames to datasets.Dataset objects
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

# Tokenize the datasets
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)

# Set up training arguments, using 'eval_strategy' instead of 'evaluation_strategy'
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch", # Corrected parameter name
    logging_dir='./logs',
    logging_steps=500, # Added for more frequent feedback during training
)

# Fine-tune the model using the Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset, # Use tokenized dataset
    eval_dataset=tokenized_val_dataset,   # Use tokenized dataset
)

# Start fine-tuning the model
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/84000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [5]:
from transformers import TrainingArguments, Trainer

# Set up training arguments, using 'eval_strategy' instead of 'evaluation_strategy'
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch", # Corrected parameter name
    logging_dir='./logs',
    logging_steps=500, # Added for more frequent feedback during training
)

# Fine-tune the model using the Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset, # Use tokenized dataset
    eval_dataset=tokenized_val_dataset   # Use tokenized dataset
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


NameError: name 'model' is not defined

#Explanation
We use BERT for sequence classification, with three output labels in this example. The training arguments are defined using the TrainingArguments class, where you can specify the batch size, number of epochs, and evaluation strategy. Trainer is used to handle the fine-tuning process by taking care of the training loop, backpropagation, and model updates.

#Step 3: Evaluate the model
After fine-tuning the model, it’s time to evaluate its performance on the test set. This allows you to measure how well the model generalizes to new, unseen data.

Instructions
Evaluate the fine-tuned model on the test set using standard metrics such as accuracy, F1 score, precision, and recall.

Analyze the results to determine how well the model performs.

Code example

In [1]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Evaluate the model on the test set
predictions_output = trainer.predict(test_data)
predictions = predictions_output.predictions.argmax(axis=-1) # Assuming a classification task

# Compute evaluation metrics
accuracy = accuracy_score(test_data['label'], predictions)
f1 = f1_score(test_data['label'], predictions, average='weighted')
precision = precision_score(test_data['label'], predictions, average='weighted')
recall = recall_score(test_data['label'], predictions, average='weighted')

print(f"Test Accuracy: {accuracy}")
print(f"Test F1 Score: {f1}")
print(f"Test Precision: {precision}")
print(f"Test Recall: {recall}")

NameError: name 'trainer' is not defined

#Explanation
The model’s performance is evaluated using four metrics: accuracy, F1 score, precision, and recall. Accuracy gives a general overview of how well the model predicts, while F1 score, precision, and recall provide a deeper analysis of the model's effectiveness, especially for unbalanced datasets.

#Step 4: Optimize the fine-tuning process (optional)
You can experiment with different training configurations and hyperparameters to further improve the model's performance. This includes adjusting the learning rate, batch size, and number of epochs. You may also use hyperparameter search techniques to find the best configuration.

Optional instructions
Adjust hyperparameters such as learning rate and batch size.

Use a hyperparameter search to find the best settings for your task.

Code example

In [ ]:
# Use hyperparameter search to optimize fine-tuning
best_model = trainer.hyperparameter_search(
    direction="maximize",
    n_trials=10
)

#Explanation
Hyperparameter tuning allows you to experiment with different configurations, automatically finding the best setup for fine-tuning based on performance metrics.